# Tracing for Different Types of Runs

### Types of Runs

LangSmith 支持多种不同的运行类型（types of Runs），您可以在 `@traceable` 装饰器中指定运行的类型。运行的类型包括：
- LLM: Invokes an LLM
- Retriever: Retrieves documents from databases or other sources
- Tool: Executes actions with function calls
- Chain: Default type; combines multiple Runs into a larger process
- Prompt: Hydrates a prompt to be used with an LLM
- Parser: Extracts structured data

### Setup

In [ ]:
# You can set them inline!
# import os
# os.environ["OPENAI_API_KEY"] = "your openai api key"
# os.environ["LANGSMITH_API_KEY"] = "your langsmith api key"
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_PROJECT"] = "langsmith-notebook"

In [1]:
# Or you can use a .env file
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

True

### LLM Runs for Chat Models

LangSmith 为 LLM 轨迹提供了特殊的渲染和处理功能。为了充分利用此功能，您必须以特定格式记录 LLM 轨迹。

对于聊天式模型，输入必须是 OpenAI 兼容格式的消息列表，以 Python 字典或 TypeScript 对象的形式表示。每条消息必须包含键 role 和 content。

输出结果可接受以下任何一种格式：

- 一个字典/对象，其中包含键“choices”，其值为一个字典/对象列表。每个字典/对象必须包含键“message”，该键映射到一个消息对象，该对象包含键“role”和“content”。
- 一个字典/对象，其中包含键 message，其值为一个消息对象，该对象包含键 role 和 content。
- 一个包含两个元素的元组/数组，其中第一个元素是角色，第二个元素是内容。
- 包含关键角色和内容的字典/对象。

函数的输入应该命名为 messages。

您还可以提供以下元数据字段，以帮助 LangSmith 识别模型并计算成本。如果使用 LangChain 或 OpenAI 封装器，这些字段将自动正确填充。
- ls_provider: The provider of the model, eg "openai", "anthropic", etc.
- ls_model_name: The name of the model, eg "gpt-5.1", "claude-opus-4-5-20251101", etc.

In [3]:
from langsmith import traceable

inputs = [
  {"role": "system", "content": "You are a helpful assistant."},
  {"role": "user", "content": "I'd like to book a table for two."},
]

output = {
  "choices": [
      {
          "message": {
              "role": "assistant",
              "content": "Sure, what time would you like to book the table for?"
          }
      }
  ]
}

# Can also use one of:
# output = {
#     "message": {
#         "role": "assistant",
#         "content": "Sure, what time would you like to book the table for?"
#     }
# }
#
# output = {
#     "role": "assistant",
#     "content": "Sure, what time would you like to book the table for?"
# }
#
# output = ["assistant", "Sure, what time would you like to book the table for?"]

@traceable(
  # TODO: Add an run_type="llm", and metadata for ls_provider, and ls_model_name
  run_type="llm",
  metadata={"ls_provider": "qwen", "ls_model_name": "qwen3-max"}
)
def chat_model(messages: list):
  return output

chat_model(inputs)

{'choices': [{'message': {'role': 'assistant',
    'content': 'Sure, what time would you like to book the table for?'}}]}

### Handling Streaming LLM Runs

对于流式传输，您可以将输出“简化”成与非流式传输版本相同的格式。目前只有 Python 支持此功能。

In [4]:
def _reduce_chunks(chunks: list):
    all_text = "".join([chunk["choices"][0]["message"]["content"] for chunk in chunks])
    return {"choices": [{"message": {"content": all_text, "role": "assistant"}}]}

@traceable(
    run_type="llm",
    metadata={"ls_provider": "my_provider", "ls_model_name": "my_model"},
    # TODO: Add a reduce_fn
    reduce_fn=_reduce_chunks
)
def my_streaming_chat_model(messages: list):
    for chunk in ["Hello, " + messages[1]["content"]]:
        yield {
            "choices": [
                {
                    "message": {
                        "content": chunk,
                        "role": "assistant",
                    }
                }
            ]
        }

list(
    my_streaming_chat_model(
        [
            {"role": "system", "content": "You are a helpful assistant. Please greet the user."},
            {"role": "user", "content": "polly the parrot"},
        ],
    )
)

[{'choices': [{'message': {'content': 'Hello, polly the parrot',
     'role': 'assistant'}}]}]

### Retriever Runs + Documents

许多 LLM 应用需要从向量数据库、知识图谱或其他类型的索引中查找文档。检索器轨迹用于记录检索器检索到的文档。LangSmith 为轨迹中的检索步骤提供了特殊的渲染方式，以便更轻松地理解和诊断检索问题。为了正确渲染检索步骤，需要执行以下几个步骤。

1. 使用 run_type="retriever" 注释检索器步骤。
2. 从检索步骤返回 Python 字典或 TypeScript 对象列表。每个字典应包含以下键：
    - page_content: 文档的文本内容。
    - type: 这应该始终是“文档”。
    - metadata: 包含文档元数据的 Python 字典或 TypeScript 对象。这些元数据将显示在跟踪信息中。

In [5]:
from langsmith import traceable

def _convert_docs(results):
  return [
      {
          "page_content": r,
          "type": "Document",
          "metadata": {"foo": "bar"}
      }
      for r in results
  ]

@traceable(
    # TODO: Add an run_type="retriever"
    run_type="retriever"
)
def retrieve_docs(query):
  # Retriever returning hardcoded dummy documents.
  # In production, this could be a real vector datatabase or other document index.
  contents = ["Document contents 1", "Document contents 2", "Document contents 3"]
  return _convert_docs(contents)

retrieve_docs("User query")

[{'page_content': 'Document contents 1',
  'type': 'Document',
  'metadata': {'foo': 'bar'}},
 {'page_content': 'Document contents 2',
  'type': 'Document',
  'metadata': {'foo': 'bar'}},
 {'page_content': 'Document contents 3',
  'type': 'Document',
  'metadata': {'foo': 'bar'}}]

### Tool Calling

LangSmith 对模型发起的工具调用进行了自定义渲染，以便清楚地显示何时使用了提供的工具。

In [7]:
from langsmith import traceable
from openai import OpenAI
from typing import List, Optional
import json
import os

openai_client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

@traceable(
  # TODO: Add an run_type="tool"
  run_type="tool"
)
def get_current_temperature(location: str, unit: str):
    return 65 if unit == "Fahrenheit" else 17

@traceable(run_type="llm")
def call_openai(
    messages: List[dict], tools: Optional[List[dict]]
) -> str:
  return openai_client.chat.completions.create(
    model="qwen3-max",
    messages=messages,
    temperature=0,
    tools=tools
  )

@traceable(run_type="chain")
def ask_about_the_weather(inputs, tools):
  response = call_openai(inputs, tools)
  tool_call_args = json.loads(response.choices[0].message.tool_calls[0].function.arguments)
  location = tool_call_args["location"]
  unit = tool_call_args["unit"]
  tool_response_message = {
    "role": "tool",
    "content": json.dumps({
        "location": location,
        "unit": unit,
        "temperature": get_current_temperature(location, unit),
    }),
    "tool_call_id": response.choices[0].message.tool_calls[0].id
  }
  inputs.append(response.choices[0].message)
  inputs.append(tool_response_message)
  output = call_openai(inputs, None)
  return output

tools = [
    {
      "type": "function",
      "function": {
        "name": "get_current_temperature",
        "description": "Get the current temperature for a specific location",
        "parameters": {
          "type": "object",
          "properties": {
            "location": {
              "type": "string",
              "description": "The city and state, e.g., San Francisco, CA"
            },
            "unit": {
              "type": "string",
              "enum": ["Celsius", "Fahrenheit"],
              "description": "The temperature unit to use. Infer this from the user's location."
            }
          },
          "required": ["location", "unit"]
        }
      }
    }
]
inputs = [
  {"role": "system", "content": "You are a helpful assistant."},
  {"role": "user", "content": "What is the weather today in New York City?"},
]

ask_about_the_weather(inputs, tools)

ChatCompletion(id='chatcmpl-215b215c-1017-4c92-9907-e72bcbcbc27f', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The current temperature in New York City is 65°F.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1765113197, model='qwen3-max', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=13, prompt_tokens=104, total_tokens=117, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=0)))